##Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler

##Load Datasets

In [ ]:
high = pd.read_csv("/content/maternal_high_burden.csv")
moderate = pd.read_csv("/content/maternal_moderate_burden.csv")
low = pd.read_csv("/content/maternal_low_burden.csv")

In [ ]:
print(high.columns)
print(moderate.columns)
print(low.columns)

##Merge Datasets

In [ ]:
import pandas as pd

high = pd.read_csv("/content/maternal_high_burden.csv")
moderate = pd.read_csv("/content/maternal_moderate_burden.csv")
low = pd.read_csv("/content/maternal_low_burden.csv")

df = pd.concat([high, moderate, low], ignore_index=True)

print(df.shape)
df.head()

In [ ]:
df.to_csv("maternal_health_combined.csv", index=False)

##Data Understanding

In [ ]:
df.shape
df.info()
df.isnull().sum()

##Data Cleaning

In [ ]:
df.drop("id", axis=1, inplace=True)

##Risk Level Distribution

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x="risk_level", data=df)

plt.title("Distribution of Pregnancy Risk Levels")
plt.show()

In [ ]:
risk_counts = df['risk_level'].value_counts()

plt.figure(figsize=(6,6))

plt.pie(
risk_counts,
labels=risk_counts.index,
autopct='%1.1f%%'
)

plt.title("Risk Level Percentage")

plt.show()


##Numerical Feature Distributions

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns

for col in numeric_cols:
    plt.figure()
    sns.histplot(df[col], kde=True)
    plt.title(col + " Distribution")
    plt.show()

##Scatter Plots

In [ ]:
plt.figure(figsize=(7,5))

sns.scatterplot(
x='bmi_pre_pregnancy',
y='fasting_glucose_mgdl',
hue='risk_level',
data=df
)

plt.title("BMI vs Glucose by Risk Level")
plt.show()

In [ ]:
sns.scatterplot(
x='age_years',
y='bmi_pre_pregnancy',
hue='risk_level',
data=df
)

plt.title("Age vs BMI")
plt.show()

##Feature Analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_features = [
    'age_years',
    'gravidity',
    'parity',
    'gestational_age_weeks',
    'bmi_pre_pregnancy',
    'systolic_bp_mmhg',
    'diastolic_bp_mmhg',
    'hemoglobin_gdl',
    'fasting_glucose_mgdl',
    'anc_visits'
]

categorical_features = [
    'proteinuria',
    'hiv_status'
]

##Numerical Features vs Risk Level

In [ ]:
for feature in numeric_features:
    plt.figure(figsize=(8,5))
    sns.barplot(x='risk_level', y=feature, data=df, errorbar=None)
    plt.title(f"Average {feature} by Risk Level")
    plt.xlabel("Risk Level")
    plt.ylabel(feature)
    plt.tight_layout()
    plt.show()

##Categorical Features vs Risk Level

In [ ]:
for feature in categorical_features:
    plt.figure(figsize=(8,5))
    sns.countplot(x=feature, hue='risk_level', data=df)
    plt.title(f"Distribution of {feature} by Risk Level")
    plt.xlabel(feature)
    plt.ylabel("Count")
    plt.legend(title='Risk Level')
    plt.tight_layout()
    plt.show()

##Outlier Detection

In [ ]:
for col in numeric_features:
    sns.boxplot(x=df[col])
    plt.show()


##Correlation Analysis

In [ ]:
plt.figure(figsize=(12, 10))

correlation_matrix = df.corr(numeric_only=True)

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    vmin=-1,
    vmax=1
)

plt.title("Correlation Matrix of Maternal Health Features")
plt.tight_layout()
plt.show()

##Feature Engineering

In [ ]:
# Feature Engineering

df['bp_diff'] = (
    df['systolic_bp_mmhg']
    - df['diastolic_bp_mmhg']
)

df.head()

##Remove Unneeded Columns

In [ ]:
df.drop(
    [
        'delivery_mode',
        'pregnancy_outcome',
        'primary_complication'
    ],
    axis=1,
    inplace=True
)

##Outlier Treatment

In [ ]:
numeric_cols = [
    'age_years',
    'gravidity',
    'parity',
    'gestational_age_weeks',
    'bmi_pre_pregnancy',
    'systolic_bp_mmhg',
    'diastolic_bp_mmhg',
    'hemoglobin_gdl',
    'fasting_glucose_mgdl',
    'anc_visits',
    'bp_diff'
]

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

for col in numeric_cols:
    sns.boxplot(x=df[col])
    plt.show()


##Encoding

In [ ]:
df['anemia_status'] = df['anemia_status'].map({
    'none': 0,
    'mild': 1,
    'moderate': 2,
    'severe': 3
})
df['risk_level'] = df['risk_level'].map({
    'low': 0,
    'moderate': 1,
    'high': 2
})

##Feature & Target Selection

In [ ]:
X = df.drop('risk_level', axis=1)
y = df['risk_level']

##Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

##Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

##Random Forest Training

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

##Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

##Confusion Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

##Feature Importance

In [ ]:
import pandas as pd

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

importance = importance.sort_values(by='Importance', ascending=False)

print(importance)
plt.figure(figsize=(8,5))
sns.barplot(data=importance, x='Importance', y='Feature')
plt.title("Feature Importance")
plt.show()

##Save Model

In [ ]:
import joblib

joblib.dump(rf_model, "maternal_risk_model.pkl")
joblib.dump(scaler, "scaler.pkl")

##Class Distribution After Encoding

In [ ]:
print(df['anemia_status'].value_counts())
print(df['risk_level'].value_counts())

##Cross Validation

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

rf_cv = RandomForestClassifier(n_estimators=200, random_state=42)

pipeline = make_pipeline(StandardScaler(), rf_cv)

scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

print("Scores:", scores)
print("Mean Accuracy:", scores.mean())
print("Std:", scores.std())

##Model Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

lr_model = LogisticRegression(max_iter=1000)
dt_model = DecisionTreeClassifier()

results = {}

# Logistic Regression (needs scaling)
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
results["Logistic Regression"] = accuracy_score(y_test, lr_pred)

# Decision Tree (no scaling)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)
results["Decision Tree"] = accuracy_score(y_test, dt_pred)

# Random Forest (no scaling)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
results["Random Forest"] = accuracy_score(y_test, rf_pred)

results_df = pd.DataFrame(results.items(), columns=["Model", "Accuracy"])
print(results_df)

##Model Comparison Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.barplot(data=results_df, x="Model", y="Accuracy")
plt.title("Model Comparison")
plt.xticks(rotation=20)
plt.show()